# 03 — QLoRA Fine-Tuning: Training TinyLlama on Alpaca

**Project**: LLM Fine-Tuning with QLoRA  
**Notebook**: 3 of 6  
**GPU required**: ✅ Yes — T4 GPU (free tier)  
**Runtime**: ~35–45 minutes

---

## What This Notebook Does

This is the core of the entire project. We:
1. Load TinyLlama-1.1B in 4-bit quantization
2. Inject LoRA adapter matrices into attention layers
3. Verify only ~0.38% of parameters are trainable
4. Run supervised fine-tuning on Alpaca for 3 epochs
5. Monitor the loss curve in real time
6. Save the LoRA adapter weights

## The QLoRA Stack (visualized)

```
TinyLlama-1.1B (frozen, 4-bit NF4)      ~0.6 GB VRAM
         +
LoRA Adapters (trainable, float16)       ~30 MB VRAM
         +
Gradient Checkpointing                  reduces peak memory
         +
Paged AdamW Optimizer                   memory-efficient
         =
Total VRAM usage: ~5–7 GB  ✅  (fits T4's 15 GB)
```

## Key Concepts in This Notebook
- LoRA adapter injection and what it looks like in the model
- The PyTorch training loop (demystified)
- What training loss tells you
- How to read a loss curve
- Saving LoRA adapters vs full model weights

---
## ⚙️ Cell 1: Install All Dependencies

In [ ]:
# This installs everything needed for QLoRA fine-tuning
# bitsandbytes: 4-bit quantization
# peft: LoRA adapter injection
# trl: SFTTrainer (supervised fine-tuning)
# accelerate: GPU device management
!pip install -q transformers==4.36.0 peft==0.7.0 trl==0.7.4 \
                datasets bitsandbytes accelerate \
                matplotlib pyyaml

print("✅ All dependencies installed")

In [ ]:
import os, sys

GITHUB_USERNAME = "YOUR_USERNAME"   # ← change this
REPO_NAME = "llm-finetuning-peft"

if not os.path.exists(REPO_NAME):
    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
else:
    !cd {REPO_NAME} && git pull

sys.path.insert(0, f"/content/{REPO_NAME}/src")
os.makedirs(f"/content/{REPO_NAME}/results", exist_ok=True)

# Change working directory so relative paths in train.py work correctly
os.chdir(f"/content/{REPO_NAME}")
print(f"✅ Working directory: {os.getcwd()}")

---
## 🖥️ Cell 2: GPU Check + Memory Baseline

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU found! Enable T4 GPU before running this notebook.")

gpu = torch.cuda.get_device_properties(0)
total_vram = gpu.total_memory / 1e9

print(f"✅ GPU: {gpu.name}")
print(f"   VRAM: {total_vram:.1f} GB")
print(f"   PyTorch: {torch.__version__}")
print()
print("📊 Expected memory usage during training:")
print(f"   Model (4-bit NF4) : ~0.6 GB")
print(f"   LoRA adapters     : ~0.03 GB")
print(f"   Activations       : ~2–3 GB")
print(f"   Optimizer states  : ~0.5 GB")
print(f"   Peak estimate     : ~5–7 GB")
print(f"   Available VRAM    : {total_vram:.1f} GB  ✅")

---
## 📥 Cell 3: Load Dataset

Same split as notebook 01 — `seed=42` ensures identical examples.

In [ ]:
from data_utils import load_alpaca_dataset

train_dataset, test_dataset = load_alpaca_dataset(
    train_size=2000,
    test_size=200,
    seed=42
)

print(f"\n✅ Dataset ready:")
print(f"   Train: {len(train_dataset):,} examples")
print(f"   Test : {len(test_dataset):,} examples")

---
## 🤖 Cell 4: Load TinyLlama in 4-bit

Same as notebook 02, but now we follow it immediately with LoRA injection.

In [ ]:
from model_utils import load_tokenizer, load_base_model

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = load_tokenizer(MODEL_NAME)
model     = load_base_model(MODEL_NAME)

# Check GPU memory after loading
used = torch.cuda.memory_allocated() / 1e9
print(f"\n📊 GPU memory after base model load: {used:.2f} GB")

---
## 💉 Cell 5: Inject LoRA Adapters — The Key Step

This is where PEFT happens. Let's go deep on what this does.

**Before LoRA injection**: model has 1.1B parameters, ALL frozen, nothing is trainable.  
**After LoRA injection**: model has ~4M NEW trainable parameters added (the A and B matrices).

The base model weights **never change**. We only train the tiny adapters.

In [ ]:
from model_utils import get_lora_config, apply_lora, print_trainable_parameters

# ── Configure LoRA ─────────────────────────────────────────────────────────
LORA_R = 8   # ← rank. Change to 4 or 16 for ablation study

lora_config = get_lora_config(r=LORA_R)

print(f"📋 LoRA Configuration:")
print(f"   Rank (r)         : {lora_config.r}")
print(f"   Alpha            : {lora_config.lora_alpha}")
print(f"   Target modules   : {lora_config.target_modules}")
print(f"   Dropout          : {lora_config.lora_dropout}")
print(f"   Task type        : {lora_config.task_type}")

In [ ]:
# ── Apply LoRA to the model ────────────────────────────────────────────────
model = apply_lora(model, lora_config)

# ── The magic moment — print trainable parameters ─────────────────────────
print_trainable_parameters(model)

In [ ]:
# ── Inspect what LoRA actually added to the model ─────────────────────────
# Look at one attention layer — you'll see the lora_A and lora_B matrices

print("🔬 Inspecting LoRA injection in layer 0 (q_proj):")
print("-" * 60)
q_proj = model.model.layers[0].self_attn.q_proj
print(q_proj)
print()
print("📌 Notice:")
print("   'base_layer' = the original frozen weight matrix")
print("   'lora_A'     = the A adapter matrix (input projection)")
print("   'lora_B'     = the B adapter matrix (output projection)")
print("   Only lora_A and lora_B have requires_grad=True")

print()
print("🔢 Adapter matrix sizes:")
for name, param in q_proj.named_parameters():
    print(f"   {name:<30}: shape={tuple(param.shape)}, trainable={param.requires_grad}")

---
## ⚙️ Cell 6: Training Configuration

We load hyperparameters from `configs/qlora_config.yaml` — single source of truth.  
Let's review what each argument does before we train.

In [ ]:
import yaml
from train import get_training_args, load_config

config = load_config("configs/qlora_config.yaml")

# Print the training section of config
print("📋 Training Hyperparameters (from qlora_config.yaml):")
print("-" * 55)
for key, val in config["training"].items():
    print(f"   {key:<35}: {val}")

print()
batch = config["training"]["per_device_train_batch_size"]
accum = config["training"]["gradient_accumulation_steps"]
epochs = config["training"]["num_train_epochs"]
steps_per_epoch = len(train_dataset) // (batch * accum)
total_steps = steps_per_epoch * epochs

print(f"📊 Derived Training Stats:")
print(f"   Effective batch size  : {batch} × {accum} = {batch * accum} examples")
print(f"   Steps per epoch       : {steps_per_epoch}")
print(f"   Total training steps  : {total_steps}")
print(f"   Warmup steps (3%)     : {int(total_steps * 0.03)}")

In [ ]:
from transformers import TrainingArguments

training_args = get_training_args(config)
training_args.output_dir = f"outputs/qlora-r{LORA_R}"
os.makedirs(training_args.output_dir, exist_ok=True)

print(f"✅ Training arguments configured")
print(f"   Checkpoints will be saved to: {training_args.output_dir}")

---
## 🏋️ Cell 7: Create SFTTrainer and Train!

**SFTTrainer** (Supervised Fine-Tuning Trainer) from TRL wraps the PyTorch training loop.  
Under the hood, every step it does:
```
forward pass → compute loss → backward pass → optimizer step → zero gradients
```
Watch the loss values in the output. They should trend **downward**.  
This is the model learning — in real time.

⏱️ **Expected time**: 35–45 minutes on T4 free tier

In [ ]:
from trl import SFTTrainer

print("🏋️  Creating SFTTrainer...")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    dataset_text_field="text",    # Column that contains our formatted prompts
    max_seq_length=512,           # Truncate examples longer than 512 tokens
    args=training_args,
)

print("✅ Trainer ready!")
print()
print("📌 What you'll see during training:")
print("   {'loss': X.XX, 'learning_rate': X.XX, 'epoch': X.XX}")
print("   - loss     : how wrong the model is. Should decrease each line.")
print("   - lr       : current learning rate (follows cosine schedule)")
print("   - epoch    : progress through the dataset (0→1→2→3)")
print()
print("⏱️  Starting training now... (~35-45 min on T4)")
print("=" * 60)

In [ ]:
# ── THE TRAINING RUN ──────────────────────────────────────────────────────
# This cell runs for ~35-45 minutes. Do not interrupt.
# You'll see loss values logged every 10 steps.

train_result = trainer.train()

print()
print("=" * 60)
print("✅ TRAINING COMPLETE!")
print("=" * 60)
print(f"   Final training loss  : {train_result.training_loss:.4f}")
print(f"   Total steps          : {train_result.global_step}")
print(f"   Training time        : {train_result.metrics['train_runtime'] / 60:.1f} minutes")
print(f"   Samples/second       : {train_result.metrics.get('train_samples_per_second', 'N/A')}")

---
## 📉 Cell 8: Plot the Training Loss Curve

Visualize how the loss changed over training. This is the first result of your project.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract loss logs from trainer history
logs  = trainer.state.log_history
steps  = [l["step"] for l in logs if "loss" in l]
losses = [l["loss"]  for l in logs if "loss" in l]

fig, ax = plt.subplots(figsize=(11, 5))

# Raw loss line
ax.plot(steps, losses, color="#94A3B8", linewidth=1, alpha=0.7, label="Loss (per step)")

# Smoothed trend
window = max(5, len(losses) // 20)
smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
ax.plot(steps[window - 1:], smoothed, color="#4F46E5",
        linewidth=2.5, label=f"Smoothed trend")

# Epoch markers
epoch_steps = [int(len(train_dataset) / (config['training']['per_device_train_batch_size'] *
               config['training']['gradient_accumulation_steps']) * i)
               for i in range(1, config['training']['num_train_epochs'] + 1)]
for ep_step in epoch_steps:
    if ep_step <= max(steps):
        ax.axvline(ep_step, color="#F59E0B", linestyle=":", alpha=0.8, linewidth=1.5)
        ax.text(ep_step + 1, max(losses) * 0.98,
                f"Ep end", fontsize=8, color="#F59E0B", rotation=90)

# Annotations
ax.annotate(f"Start: {losses[0]:.3f}", xy=(steps[0], losses[0]),
            xytext=(steps[0] + max(steps)*0.04, losses[0] + 0.05),
            fontsize=10, color="#64748B",
            arrowprops=dict(arrowstyle="->", color="#64748B"))
ax.annotate(f"End: {losses[-1]:.3f}", xy=(steps[-1], losses[-1]),
            xytext=(steps[-1] - max(steps)*0.2, losses[-1] + 0.1),
            fontsize=10, color="#4F46E5",
            arrowprops=dict(arrowstyle="->", color="#4F46E5"))

improvement = (losses[0] - losses[-1]) / losses[0] * 100

ax.set_xlabel("Training Step", fontsize=12)
ax.set_ylabel("Cross-Entropy Loss", fontsize=12)
ax.set_title(
    f"Training Loss — TinyLlama-1.1B QLoRA (r={LORA_R}) on Alpaca\n"
    f"Loss reduced by {improvement:.1f}% over {train_result.global_step} steps",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_facecolor("#F8FAFC")

plt.tight_layout()
plt.savefig("results/training_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n📉 Loss curve saved to results/training_loss_curve.png")
print(f"\n💡 How to read this plot:")
print(f"   - Steady downward trend ✅ = model is learning")
print(f"   - Sudden spikes = learning rate may be too high")
print(f"   - Plateau early = may need more epochs or higher LR")
print(f"   - Your total loss reduction: {improvement:.1f}%")

---
## 💾 Cell 9: Save the LoRA Adapter

**Critical distinction**: we save ONLY the adapter — not the full model.  
The base model (1.1B params) is publicly available on HuggingFace Hub — no need to store it.  
We only need to store what WE trained: the LoRA adapter (~50 MB).

In [ ]:
import os

ADAPTER_SAVE_PATH = f"outputs/qlora-adapter-r{LORA_R}"
os.makedirs(ADAPTER_SAVE_PATH, exist_ok=True)

print(f"💾 Saving LoRA adapter to: {ADAPTER_SAVE_PATH}")
print("   (Saving ONLY the adapter matrices — NOT the 1.1B base model weights)")

# This saves:
#   adapter_config.json   — LoRA configuration
#   adapter_model.bin     — The actual A and B weight matrices
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

# Check file sizes
print(f"\n📁 Files saved:")
total_size = 0
for fname in sorted(os.listdir(ADAPTER_SAVE_PATH)):
    fsize = os.path.getsize(os.path.join(ADAPTER_SAVE_PATH, fname))
    total_size += fsize
    print(f"   {fname:<40} {fsize / 1e6:.2f} MB")

print(f"\n   Total adapter size: {total_size / 1e6:.2f} MB")
print(f"   Compare to full model float16: ~2,200 MB")
print(f"   Adapter is {total_size / 22e8 * 100:.1f}% of the full model size — that's PEFT!")

In [ ]:
# ── Save training metrics to YAML ─────────────────────────────────────────
import yaml

metrics = {
    "model": MODEL_NAME,
    "lora_rank": LORA_R,
    "lora_alpha": LORA_R * 2,
    "train_samples": len(train_dataset),
    "epochs": config["training"]["num_train_epochs"],
    "final_loss": round(train_result.training_loss, 4),
    "initial_loss": round(losses[0], 4),
    "loss_reduction_pct": round(improvement, 2),
    "total_steps": train_result.global_step,
    "training_time_minutes": round(train_result.metrics["train_runtime"] / 60, 1),
}

metrics_path = f"{ADAPTER_SAVE_PATH}/training_metrics.yaml"
with open(metrics_path, "w") as f:
    yaml.dump(metrics, f, default_flow_style=False)

print("📊 Training Metrics Summary:")
print("-" * 40)
for k, v in metrics.items():
    print(f"   {k:<28}: {v}")

print(f"\n💾 Metrics saved to: {metrics_path}")

---
## 🧪 Cell 10: Quick Sanity Check — Does the Fine-Tuned Model Respond?

Before closing this notebook, let's do a quick test to confirm the model works.

In [ ]:
from inference import FineTunedModel, INFERENCE_PROMPT

print("🧪 Quick sanity check — testing the fine-tuned model...")
print("(Using the already-loaded model in memory — no reload needed)\n")

# Test prompts — deliberately chosen to match Alpaca training distribution
test_instructions = [
    "Give three tips for being more productive at work.",
    "Explain what a neural network is in simple terms.",
]

model.eval()

for instruction in test_instructions:
    prompt = INFERENCE_PROMPT.format(instruction=instruction)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()

    print(f"📌 Instruction: {instruction}")
    print(f"🤖 Response:\n{response}")
    print("-" * 70)

---
## 📤 Cell 11: Commit Results to GitHub

In [ ]:
# Push the loss curve and metrics (NOT the model weights — too large)
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"

!git add results/training_loss_curve.png
!git add outputs/qlora-adapter-r8/training_metrics.yaml
!git commit -m "results: add training loss curve and metrics (r=8)"
!git push

print("\n✅ Results committed to GitHub!")
print("   (Large adapter .bin file is excluded by .gitignore)")

---
## ✅ Summary

In this notebook you:

| Step | What happened |
|---|---|
| Loaded model | TinyLlama-1.1B in 4-bit (~0.6 GB VRAM) |
| Injected LoRA | ~4M trainable params added (0.38% of total) |
| Inspected adapters | Saw lora_A + lora_B inside q_proj layer |
| Configured training | Loaded hyperparams from YAML config |
| Trained model | ~35-45 min, 3 epochs, watched loss decrease |
| Plotted loss curve | Saved to `results/training_loss_curve.png` |
| Saved adapter | ~50 MB vs 2,200 MB for full model |
| Sanity checked | Fine-tuned model responded to instructions |

### Key PyTorch Concepts Covered
- `requires_grad` — which parameters get updated
- Forward pass → Loss → Backward pass → Optimizer step
- `model.eval()` — inference vs training mode
- Gradient accumulation — simulating larger batches
- Cross-entropy loss — measuring prediction quality

---
### ▶️ Next: `04_evaluation.ipynb`
We load the fine-tuned adapter, run the same 5 baseline prompts, compute ROUGE scores,
and generate all comparison charts. This is where you see the full before vs. after picture.